# 04 — Forecasting models

**Phase 4 deliverable:** all 18 upstream architectures scored against `NaiveLag` on KBANK in one
command.

18 notebooks, 4 families. The upstream `deep-learning/` set is one train loop with
`{cell} × {bidirectional} × {paths} × {decoder}` — so it is implemented once and configured 18
times. Same coverage, roughly a fifth of the code, and "compare everything on KBANK" becomes a
single command instead of 18 manual runs.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
from stock_retrofit.config import all_model_specs
from stock_retrofit.models import registered_kinds

print("model families:", ", ".join(registered_kinds()), "\n")
for spec in all_model_specs():
    print(f"  {spec.name:32s} {spec.kind:28s} <- {spec.upstream}")

## Run the whole catalogue

Equivalent to:

```bash
python -m stock_retrofit.cli evaluate --all --symbol KBANK
```

`NaiveLag` is inserted automatically whether or not you ask for it, and pinned to the top of the
table (spec R8).

In [ ]:
from stock_retrofit.report import evaluate_symbol

models = evaluate_symbol("KBANK")
models

## Reading the result

The `beats_naive` column is computed, not interpreted: **MASE < 1.00 beats the naive lag.**

Watch `dir_acc` too. These models sit around 40–45% directional accuracy — *below* a coin flip.
That is not a bug in the harness; it is what happens when a model trained to minimise squared
error on a near-random-walk return series is asked to call direction.

Note also the gap between `sharpe_gross` and `sharpe_net`. A model with a healthy frictionless
Sharpe and a negative net one has found a signal too small to pay for its own turnover — which is
the most common way a backtest lies.

In [ ]:
from stock_retrofit.eval import summarise_beats

summary = summarise_beats(models)
print(f"{summary['beat_naive']} of {summary['ran']} models beat NaiveLag on MASE")
print("winners:", summary["winners"] or "none")

worst = models.loc[models["status"] == "ok"].nsmallest(5, "sharpe_net")[
    ["model", "MASE", "dir_acc", "sharpe_gross", "sharpe_net", "turnover"]]
print("\nlargest cost drag:")
worst